In [ ]:
from typing import TypedDict ,List ,Dict ,Any, Optional ,Literal
from pydantic import BaseModel ,Field,ConfigDict

In [4]:

EQUIPMENT_TAG_PATTERN = r"^[A-Z0-9]+-[A-Z]+-[0-9]{2}[A-Z]?$"

class Equipment(BaseModel):
    model_config=ConfigDict(extra="forbid")

    id :str =Field(description="Unique identifier of this equipment.")
    tag :str =Field(
        ...,
        pattern=EQUIPMENT_TAG_PATTERN,
        description=(
            "Equipment tag. Must match exactly: "
            "[A-Z0-9]+-[A-Z]+-[0-9]{2}[A-Z]?"
        )
    )
    area:str=Field(...,description="Area identifier. It is the first part of the equipment tag.",pattern=r"^[A-Z0-9]+$")
    equipment_type:str=Field(...,description="Equipment type code. It is the second part of the equipment tag.",pattern=r"^[A-Z]+$")
    equipment_name:str=Field(...,description="Human-readable name of the equipment.")
    sequence:str=Field(...,description="Equipment sequence number. It is the third part of the tag.",pattern=r"^[0-9]{2}[A-Z]?$")

In [ ]:
class Pipeline(BaseModel):
    model_config=ConfigDict(extra="forbid")
    id: str
    from_equipment: Equipment=Field(...,description="Equipment at the start of the pipeline.")
    to_equipment: Equipment=Field(...,description="Equipment at the end of the pipeline.")

In [ ]:
class PipelineChunk(BaseModel):
    id: str
    from_equipment: Equipment=Field(...,description="Equipment at the start of the pipeline.")
    to_equipment: Equipment=Field(...,description="Equipment at the end of the pipeline.")
    exit_edge: Optional[Literal["top", "bottom", "left", "right"]] = Field(
        default=None,
        description="Edge through which this pipeline visibly continues beyond this chunk, if any."
    )

In [10]:
class Instrument(BaseModel):
    model_config=ConfigDict(extra="forbid")
    id:str
    tag:str=Field(description="Instrument tag")
    attached_to:Equipment | Pipeline =Field(description="The equipment or pipeline to which the instrument is attached.")

In [ ]:
class ChunkMetadata(BaseModel):
    model_config = ConfigDict(extra="forbid")

    chunk_id: str
    row: int
    col: int
    n_rows: int 
    n_cols: int
    bbox_in_pdf: tuple[float, float, float, float]  

In [ ]:
def get_adjacent_chunk_ids(meta: ChunkMetadata, all_meta: list[ChunkMetadata]) -> dict[str, Optional[str]]:
    neighbors = {}
    directions = {"top": (-1, 0), "bottom": (1, 0), "left": (0, -1), "right": (0, 1)}
    for direction, (dr, dc) in directions.items():
        target_row, target_col = meta.row + dr, meta.col + dc
        match = next(
            (m for m in all_meta if m.row == target_row and m.col == target_col),
            None
        )
        neighbors[direction] = match.chunk_id if match else None
    return neighbors

In [ ]:
class ChunkResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    metadata: ChunkMetadata
    equipment: List[Equipment]        
    instruments: List[Instrument]     
    pipelines: List[PipelineChunk]    


class PIDState(TypedDict):
    pdf_path: str
    coarse_equipment: List[Equipment]
    coarse_pipelines: List[Pipeline]      
    chunk_results: List[ChunkResult]      
    validated_data: Optional[Dict[str, Any]]  
    errors: List[str]

In [12]:
from dataclasses import dataclass

In [13]:
@dataclass(frozen=True)
class GridConfig:
    n_rows: int = 4
    n_cols: int = 3
    overlap_px: int = 128
    dpi: int = 200
    min_acceptable_patch_px: int = 800
    max_acceptable_patch_px: int = 2200

In [ ]:
class Chunk(BaseModel):
    metadata: ChunkMetadata
    image_path: str

In [14]:
from pdf2image import convert_from_path
from PIL import Image
def render_pdf_to_image(pdf_path: str, dpi: int = 200) -> Image.Image:
    pages = convert_from_path(pdf_path, dpi=dpi)
    if len(pages) != 1:
        raise ValueError(f"Expected a single-page PDF, got {len(pages)} pages: {pdf_path}")
    return pages[0]


In [ ]:
def deterministic_chunking(state: PIDState, config: GridConfig = GridConfig()) -> dict:
    image = render_pdf_to_image(state["pdf_path"], dpi=config.dpi)
    chunks, chunk_errors = chunk_image(image, config)  

    out_dir = Path("chunks") / Path(state["pdf_path"]).stem
    out_dir.mkdir(parents=True, exist_ok=True)

    saved_chunks = []
    for chunk in chunks:
        image_path = str(out_dir / f"{chunk.metadata.chunk_id}.png")
        chunk.image.save(image_path)
        saved_chunks.append(
            Chunk(metadata=chunk.metadata, image_path=image_path)
        )

    return {
        "chunks": saved_chunks,
        "errors": state.get("errors", []) + chunk_errors,
    }